In [4]:
import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections
from nighthawk.models.valuation import node_price_predictor
import time
import pickle
import dill
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
print("Here the job starts!")
print(str(datetime.now()))
try:
    sys.path.append("/var/www/python/Prod/nighthawk/")
except:
    print("oh..this is in kubernetes")
from nighthawk.models.valuation import ve_model_functions
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sys, os, inspect, shutil, importlib
sys.path.insert(0, '/var/www/python/Qingcheng/Fourier')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections, sql_functions, dataframe_functions
from nighthawk.data.pipeline.var_handler import loadwindgen_vh, wind_vh
from nighthawk.data.pipeline.common_functions import wind
from google.cloud import bigquery
import utils_fourier.ve_portfolio_constructor_fourier as ve_portfolio_constructor_fourier
from common_functions import get_scale_factor
from nighthawk.data.network.node import Node


Here the job starts!
2026-04-16 16:04:46.992369


In [ ]:
def NN_training_module_shuffle(node_num,dt):
    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    # node_num = 1005
    # dt = '2026-01-22'

    opexchange     = "SPP"
    data_location  = "training"  # 1=training, 2=secondRun
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    class DNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
            super(DNN, self).__init__()
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size)
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            return self.fc3(out)


    class QuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
            super(QuantileDNN, self).__init__()
            self.quantiles = quantiles
            self.output_size = output_size
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            x = self.fc3(out)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=0.25, random_state=42)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/3, random_state=42)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator =g )
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
        mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
        quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break


        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)
        
        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

    return valuation_models 

def NN_training_module_no_shuffle(node_num,dt):
    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    # node_num = 1005
    # dt = '2026-01-22'

    opexchange     = "SPP"
    data_location  = "training"  # 1=training, 2=secondRun
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    class DNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
            super(DNN, self).__init__()
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size)
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            return self.fc3(out)


    class QuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
            super(QuantileDNN, self).__init__()
            self.quantiles = quantiles
            self.output_size = output_size
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            x = self.fc3(out)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=0.25, random_state=42)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/3, random_state=42)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
        mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
        quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break


        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)
        
        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

    return valuation_models 


Here the job starts!
2026-04-15 20:54:21.976347
Early stopping at epoch 158


In [ ]:
def fourier_port(df, start_date='2023-01-01', end_date='2026-09-01', saved='temp_file'):
    def hourly_lwg_cut(df, bid_date):
        total_lwg_df = loadwindgen_vh.get_data_and_mapping_for_total_lwg(
            [636], "SPP", start_dt=bid_date, end_dt=bid_date, n_day_pctl_flag=False)[0]
        high_lwg_hr_lst = list(total_lwg_df.loc[total_lwg_df['spp_loadwindgen_forecast_f'] >= 55000, 'hr'].unique())
        print(f"lwg cut hours: {high_lwg_hr_lst}")
        if len(high_lwg_hr_lst) > 0:
            df['bid_mw'] = np.where(df['hr'].isin(high_lwg_hr_lst), 0, df['bid_mw'])
        return df

    def apply_price_filter(df):
        df['bid_price'] = np.where((df['incdec'] == 'Increment') & (df['bid_price'] < -100), -100, df['bid_price'])
        df['bid_price'] = np.where((df['incdec'] == 'Decrement') & (df['bid_price'] > 300), 300, df['bid_price'])
        return df[df['bid_price'] < 2000]

    def apply_mw_filter(df):
        return df[df['bid_mw'] >= 0.1]

    def portfolio_cut(portfolio, bid_date):
        portfolio = apply_price_filter(portfolio)
        portfolio = hourly_lwg_cut(portfolio, bid_date)
        portfolio = apply_mw_filter(portfolio)
        return portfolio

    # --- config ---
    opexchange    = 'SPP'
    run_number    = 1
    condition_label = 2  # 0=test, 1=holdout, 2=production
    valuationModel_label = ''
    nodeSelection_label  = 'Fourier'
    file_location_cloudserver = '/var/www/python/Qingcheng/temp_test_fourier/'
    os.makedirs(file_location_cloudserver + 'return_and_risk', exist_ok=True)
    os.makedirs(file_location_cloudserver + 'nodeSelection', exist_ok=True)
    conn = connections.get_sql_connection(database='temp')

    # --- segments (matches production) ---
    segments = pd.DataFrame({
        'quantile_DA': ['da_total_q95', 'da_total_q90', 'da_total_q80', 'da_total_q70',
                        'da_total_q60', 'da_total_q50', 'da_total_q40', 'da_total_q30',
                        'da_total_q20', 'da_total_q10', 'da_total_q5'],
        'decSegment': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
        'incSegment': [11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]})

    # --- load data ---
    df = df[df['dt'] >= start_date]
    df = df[df['dt'] <= end_date]  # test single date

    all_portfolios = []
    dates = sorted(df['dt'].unique())
    print(f'Running portfolio construction for {len(dates)} dates')

    try:
        for bid_date in dates:
            print(f'\n--- {bid_date} ---')
            try:
                # 1. Valuation model from CSV
                df_date = df[df['dt'] == bid_date].copy()
                df_date['node_num'] = df_date['node_num'].astype(int)
                df_date['dt']       = df_date['dt'].astype(str)
                df_date['hr']       = df_date['hr'].astype(int)
                total_col = [col for col in df_date.columns if
                            col.startswith('da_total_') or col.startswith('rt_total_')]
                valuationModel = df_date[['dt', 'hr', 'node_num'] + total_col].groupby(
                    ['dt', 'hr', 'node_num']).max().reset_index()

                # 2. Node selection
                nodeSelection = pd.read_sql(
                    f"SELECT * FROM Fourier_{opexchange}.nodeSelection WHERE dt = '{bid_date}' AND source = 'PCA'", conn)
                nodeSelection['dt']       = nodeSelection['dt'].astype(str)
                nodeSelection['node_num'] = nodeSelection['node_num'].astype(int)
                nodeSelection['nodeSelection'] = nodeSelection_label
                nodeSelection.to_csv(file_location_cloudserver + 'nodeSelection/' + bid_date + '.csv', index=False)

                # 3. Op_rate and ref price
                ve_port = ve_portfolio_constructor_fourier.VEPortfolioConstructorFourier(opexchange)
                valuationModel = ve_port.get_oprate_lmp_price_and_ref_value(valuationModel)
                for col in ['op_rate_inc_a', 'op_rate_dec_a', 'op_rate_inc_f', 'op_rate_dec_f', 'bid_ref_price', 'offer_ref_price']:
                    if col in valuationModel.columns:
                        valuationModel[col].fillna(value=valuationModel[col].mean(), inplace=True)

                # 4. Cumulative return and risk
                ve_port.calculate_cumulative_return_and_risk_for_one_day(
                    valuationModel,
                    save_cloudserver_location=file_location_cloudserver + 'return_and_risk/',
                    prediction='total', segments=segments, date=bid_date)

                # 5. Wind / LWG for constraints
                total_wind_df = wind_vh.get_data_and_mapping_for_baa_zonal_wind(
                    node_list=[636], opexchange='SPP',
                    start_dt=(pd.to_datetime(bid_date) - pd.to_timedelta('1D')).strftime('%Y-%m-%d'),
                    end_dt=(pd.to_datetime(bid_date) + pd.to_timedelta('1D')).strftime('%Y-%m-%d'),
                    var_spec=['f'], impute=True, ramp_flag=True, ramp_periods=[2])[0].rename(columns={
                        'e_spp_baa_zonal_wind_forecast_f': 'spp_wind_total_forecast_f',
                        'e_spp_baa_zonal_wind_actual_a': 'spp_wind_total_actual_a',
                        'BackwardRampNoSlope2_e_spp_baa_zonal_wind_forecast_f': 'BackwardRampNoSlope2_spp_wind_total_forecast_f',
                        'BackwardRampNoSlope2_e_spp_baa_zonal_wind_actual_a': 'BackwardRampNoSlope2_spp_wind_total_actual_a'})
                
                total_lwg_df = loadwindgen_vh.get_data_and_mapping_for_baa_zonal_lwg(
                    [636], "SPP",
                    start_dt=(pd.to_datetime(bid_date) - pd.to_timedelta('35D')).strftime('%Y-%m-%d'),
                    end_dt=(pd.to_datetime(bid_date) + pd.to_timedelta('1D')).strftime('%Y-%m-%d'),
                    n_day_pctl_flag=True)[0].rename(columns={
                        'e_spp_baa_zonal_loadwindgen_forecast_f': 'spp_loadwindgen_forecast_f',
                        'e_spp_baa_zonal_loadwindgen_actual_a': 'spp_loadwindgen_actual_a',
                        'Perc_30D_e_spp_baa_zonal_loadwindgen_forecast_f': 'Perc_30D_spp_loadwindgen_forecast_f',
                        'Perc_30D_e_spp_baa_zonal_loadwindgen_actual_a': 'Perc_30D_spp_loadwindgen_actual_a'})

                constraints_option = ['v2', {
                    'max_mw_per_segment': 5, 'totalRiskAllowed': 10000, 'total_mw_limit': 20000,
                    'nodal_mw_limit': 700, 'nodal_hrly_mw_limit': 60, 'nodal_hrly_incdec_mw_limit': 40,
                    'totalCollateralAllowed': 700000, 'inc_upper_perc_limit': 0.7, 'dec_upper_perc_limit': 0.7,
                    'hrly_inc_mw_limit': 450,
                    'hrly_inc_upper_perc_limit_extreme_physical_condition': {
                        'BackwardRampNoSlope2_spp_wind_total_forecast_f': -3000,
                        'hrly_inc_upper_perc_limit': 0.1, 'physical_var_df': total_wind_df},
                    'hrly_dec_upper_perc_limit_extreme_physical_condition': {
                        'Perc_30D_spp_loadwindgen_forecast_f': 0.02, 'hrly_dec_upper_perc_limit': 0.2,
                        'physical_var_df': total_lwg_df},
                }]

                # 6. Portfolio construction (production params)
                para = {
                    'maxDecPrice': 300, 'minIncPrice': -75, 'PowerROC': 1,
                    'minExpectedProfit': 0.7, 'maximumROR': 3000, 'minimumROR': 250, 'maximumROC': 100,
                    'MinExpectedReturnOnCollateral': 0, 'maxSegmentNum': 8,
                    'objectiveFunction': 'expectedProfit',
                    'constraints_option': constraints_option, 'PortionOfFullRisk': 1,
                    'segments_clear_prob': pd.DataFrame(),
                    'data_location': file_location_cloudserver,
                    'nodeSelection_label': nodeSelection_label,
                    'valuationModel_label': valuationModel_label,
                    'condition_label': condition_label, 'cumulativeRisk_ceil': -0.1
                }
                portfolio = ve_port.get_daily_terence_portfolio(bid_date, **para)

                # 7. Cut
                portfolio = portfolio[portfolio['bid_mw'] > 0]
                portfolio = portfolio_cut(portfolio, bid_date)

                # 8. Scale (fixed scale only — no wind scaling for Fourier backtest)
                scale_factor = get_scale_factor(opexchange).values[0][0]
                portfolio['bid_mw'] = portfolio['bid_mw'] * scale_factor
                portfolio['run_number'] = run_number

                print(f'  {len(portfolio)} bids, {round(portfolio["bid_mw"].sum())} total MW')
                all_portfolios.append(portfolio)

            except Exception as e:
                import traceback
                print(f'  [SKIP] {bid_date}: {e}')
                traceback.print_exc()
                continue

    finally:
        shutil.rmtree(file_location_cloudserver, ignore_errors=True)
        print(f'Cleaned up {file_location_cloudserver}')

    # --- save ---
    if all_portfolios:
        final_portfolio = pd.concat(all_portfolios, ignore_index=True)
        save_path = f'/var/www/python/Qingcheng/WFiles/{saved}.csv'
        final_portfolio.to_csv(save_path, index=False)
        print(f'\nSaved {final_portfolio.shape} to {save_path}')
    else:
        print('No portfolios generated.')


In [13]:
def eval_valuation_model(df):
    idx      = ['dt', 'hr', 'node_num']
    col_list = ['da_total_mean', 'rt_total_mean']
    df_new = (df[idx + [c for c in col_list if c in df.columns]]
                .groupby(idx, as_index=False)
                .first())
    df_new['dt'] = df_new['dt'].astype(str)

    #Get unique nodes and date range from predictions df
    node_nums = sorted(df['node_num'].dropna().unique().tolist())
    start_dt  = str(df['dt'].min())
    end_dt    = str(df['dt'].max())

    #Pull actuals from nighthawk Node (MCC=congestion, LMP=total)
    actuals_df = Node(node_nums, 'SPP').get_price(
        start_dt, end_dt,
        component=['MCC', 'LMP'],
        type=['DA', 'RT'],
        granularity='hourly'
    )[['dt', 'hr', 'node_num',  'da_total', 'rt_total']]

    actuals_df['dt']       = actuals_df['dt'].astype(str)
    actuals_df['hr']       = actuals_df['hr'].astype(int)
    actuals_df['node_num'] = actuals_df['node_num'].astype(int)

    merged = df_new.merge(actuals_df, on=idx)
    return merged

def get_metrics(df):
    # Compare with_dayzer vs without_dayzer predictions
    metrics = {}
    for target in ['da_total', 'rt_total']:
        pred = df[f'{target}_mean']
        actual = df[target]
        diff = pred - actual
        metrics[( f'{target}_ME')]  = np.mean(diff).round(2),
        metrics[( f'{target}_MSE')]  = np.mean(diff**2).round(2),
        metrics[( f'{target}_MAE')]  = np.mean(np.abs(diff)).round(2),
        metrics[( f'{target}_RMSE')] = np.sqrt(np.mean(diff ** 2)).round(2)

    met = pd.DataFrame(metrics).T
    met.columns = ['Prod']
    return met

    

In [15]:
from google.cloud import bigquery
import pandas as pd
bq_client = bigquery.Client()
df = bq_client.query("""
    SELECT *
    FROM `Darwin_Production.SPP_prediction`
    WHERE run_number = 1
      AND dt >= '2026-01-10'
      AND dt <= '2026-03-30'
    ORDER BY dt, hr, node_num
""").to_dataframe()

df_new = eval_valuation_model(df)
get_metrics(df_new)

,Prod
da_total_ME,-4.12
da_total_MSE,2018.84
da_total_MAE,19.61
da_total_RMSE,44.93
rt_total_ME,4.16
rt_total_MSE,3265.32
rt_total_MAE,27.57
rt_total_RMSE,57.14


In [54]:
bid_date   = '2026-02-01'
opexchange = 'SPP'
conn_temp = connections.get_sql_connection(database='temp')

node_sel = pd.read_sql(f"""
    SELECT dt, node_num, source, node_name, zone, broadzone
    FROM Fourier_{opexchange}.nodeSelection
    WHERE dt = '{bid_date}' AND source = 'PCA'
""", conn_temp)
node_sel['node_num'] = node_sel['node_num'].astype(int)
print(f"Total selected nodes: {len(node_sel)}")
display(node_sel)

Total selected nodes: 74


,dt,node_num,source,node_name,zone,broadzone
0,2026-02-01,8,PCA,AECC_FULTON,CSWS,SOUTH
1,2026-02-01,45,PCA,CASS_CO_2,OPPD,NORTH
2,2026-02-01,48,PCA,COFFEYVILLE_7,GRDA,SOUTH
3,2026-02-01,65,PCA,CSWFLINTCREEK1,CSWS,SOUTH
4,2026-02-01,113,PCA,CSWWELEETKA5,CSWS,SOUTH
...,...,...,...,...,...,...
69,2026-02-01,1683,PCA,KCPL.VOLT.0271,MPS,SOUTH
70,2026-02-01,1687,PCA,WR.VOLT.0254,MPS,SOUTH
71,2026-02-01,1690,PCA,WR.VOLT.0262,WR,SOUTH
72,2026-02-01,1715,PCA,CSWS.NUEN.SDM1,CSWS,SOUTH
